# Huấn luyện và Đánh giá CAE Đa Danh Mục trên Google Colab

**Đề tài:** Xây dựng và đánh giá mạng Convolutional Autoencoder trong phát hiện và khoanh vùng bất thường bề mặt sản phẩm công nghiệp.

> **Phạm vi thực nghiệm:** Đánh giá trên 3 nhóm bề mặt sản phẩm đại diện của MVTec AD:
> 1. **`bottle`** (Chai thủy tinh - Bề mặt phản quang / trong suốt)
> 2. **`capsule`** (Viên thuốc nhộng - Dược phẩm / Y tế)
> 3. **`metal_nut`** (Đai ốc cơ khí - Kim loại / Có ren xoắn)

---
## 1. Kết nối Google Drive & Kiểm tra GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi
!python -c "import torch; print('PyTorch:', torch.__version__, '| CUDA Available:', torch.cuda.is_available(), '| Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')"

---
## 2. Đồng bộ mã nguồn từ GitHub & Cài đặt thư viện

*(Kéo phiên bản code mới nhất có đủ cấu hình bottle, capsule, metal_nut)*

In [ ]:
import os

REPO_URL = "https://github.com/Trong59/industrial-anomaly-cae.git"
REPO_DIR = "/content/industrial-anomaly-cae"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull origin main

!pip install -r requirements.txt -q
print("[*] Đã đồng bộ mã nguồn và cài đặt thư viện thành công!")

---
## 3. Chuẩn bị Dữ liệu cho các Danh mục từ Google Drive

Sao chép các danh mục cần thực nghiệm sang ổ đĩa tạm `/content/` để tăng tốc độ nạp dữ liệu.

In [ ]:
import os
!mkdir -p /content/cae_data/mvtec_anomaly_detection

categories = ['bottle', 'capsule', 'metal_nut']
for cat in categories:
    drive_path = f"/content/drive/MyDrive/CAE_Project/datasets/mvtec_anomaly_detection/{cat}"
    local_path = f"/content/cae_data/mvtec_anomaly_detection/{cat}"
    if os.path.exists(drive_path) and not os.path.exists(local_path):
        print(f"[*] Đang sao chép danh mục: {cat}...")
        !cp -r "{drive_path}" "/content/cae_data/mvtec_anomaly_detection/"
        print(f"[✓] Đã sao chép xong: {cat}")
    elif os.path.exists(local_path):
        print(f"[✓] Danh mục {cat} đã có sẵn tại {local_path}")
    else:
        print(f"[!] Chưa tìm thấy {drive_path} trên Google Drive. Hãy tải thư mục {cat} lên Drive khi cần chạy!")

---
## 4. DANH MỤC 1: BOTTLE (Chai thủy tinh)

*(Bạn đã chạy xong phần này trước đó. Cell này để lưu lại nếu cần tra cứu)*

In [ ]:
# Huấn luyện, Calibrate và Evaluate E2 Bottle (Đã chạy hoàn tất)
print("[*] Danh mục BOTTLE đã hoàn thành với kết quả AUROC: 0.8032")

---
## 5. DANH MỤC 2: CAPSULE (Viên thuốc nhộng - Dược phẩm)

Mô hình sử dụng cấu hình tối ưu `configs/capsule.yaml` (L1 loss + Bottleneck nén 64 channels).

In [ ]:
# 5.1. Huấn luyện CAE trên tập bình thường của Capsule
!python train.py \
  --config configs/capsule.yaml \
  --data-root /content/cae_data/mvtec_anomaly_detection \
  --output-root /content/drive/MyDrive/CAE_Project/outputs \
  --experiment-id E2_capsule_l1_bottleneck_seed42

In [ ]:
# 5.2. Hiệu chỉnh ngưỡng cho Capsule trên tập validation normal
!python calibrate.py \
  --config configs/capsule.yaml \
  --data-root /content/cae_data/mvtec_anomaly_detection \
  --output-root /content/drive/MyDrive/CAE_Project/outputs \
  --experiment-id E2_capsule_l1_bottleneck_seed42

In [ ]:
# 5.3. Đánh giá độc lập trên tập test của Capsule
!python evaluate.py \
  --config configs/capsule.yaml \
  --data-root /content/cae_data/mvtec_anomaly_detection \
  --output-root /content/drive/MyDrive/CAE_Project/outputs \
  --experiment-id E2_capsule_l1_bottleneck_seed42 \
  --max-vis-samples 4

In [ ]:
# 5.4. Hiển thị đồ thị Loss và ROC Curve cho Capsule
import matplotlib.pyplot as plt
import os

cap_dir = "/content/drive/MyDrive/CAE_Project/outputs/E2_capsule_l1_bottleneck_seed42"
loss_path = f"{cap_dir}/plots/capsule_loss_curve.png"
roc_path = f"{cap_dir}/plots/capsule_image_roc.png"
cm_path = f"{cap_dir}/plots/capsule_confusion_matrix.png"

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
if os.path.exists(loss_path):
    axes[0].imshow(plt.imread(loss_path))
    axes[0].set_title("Capsule Loss Curve")
axes[0].axis('off')

if os.path.exists(roc_path):
    axes[1].imshow(plt.imread(roc_path))
    axes[1].set_title("Capsule ROC Curve")
axes[1].axis('off')

if os.path.exists(cm_path):
    axes[2].imshow(plt.imread(cm_path))
    axes[2].set_title("Capsule Confusion Matrix")
axes[2].axis('off')

plt.tight_layout()
plt.show()

---
## 6. DANH MỤC 3: METAL_NUT (Đai ốc kim loại - Cơ khí)

Mô hình sử dụng cấu hình tối ưu `configs/metal_nut.yaml` (L1 loss + Bottleneck nén 64 channels).

In [ ]:
# 6.1. Huấn luyện CAE trên tập bình thường của Metal Nut
!python train.py \
  --config configs/metal_nut.yaml \
  --data-root /content/cae_data/mvtec_anomaly_detection \
  --output-root /content/drive/MyDrive/CAE_Project/outputs \
  --experiment-id E2_metal_nut_l1_bottleneck_seed42

In [ ]:
# 6.2. Hiệu chỉnh ngưỡng cho Metal Nut trên tập validation normal
!python calibrate.py \
  --config configs/metal_nut.yaml \
  --data-root /content/cae_data/mvtec_anomaly_detection \
  --output-root /content/drive/MyDrive/CAE_Project/outputs \
  --experiment-id E2_metal_nut_l1_bottleneck_seed42

In [ ]:
# 6.3. Đánh giá độc lập trên tập test của Metal Nut
!python evaluate.py \
  --config configs/metal_nut.yaml \
  --data-root /content/cae_data/mvtec_anomaly_detection \
  --output-root /content/drive/MyDrive/CAE_Project/outputs \
  --experiment-id E2_metal_nut_l1_bottleneck_seed42 \
  --max-vis-samples 4

In [ ]:
# 6.4. Hiển thị đồ thị Loss và ROC Curve cho Metal Nut
import matplotlib.pyplot as plt
import os

nut_dir = "/content/drive/MyDrive/CAE_Project/outputs/E2_metal_nut_l1_bottleneck_seed42"
loss_path = f"{nut_dir}/plots/metal_nut_loss_curve.png"
roc_path = f"{nut_dir}/plots/metal_nut_image_roc.png"
cm_path = f"{nut_dir}/plots/metal_nut_confusion_matrix.png"

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
if os.path.exists(loss_path):
    axes[0].imshow(plt.imread(loss_path))
    axes[0].set_title("Metal Nut Loss Curve")
axes[0].axis('off')

if os.path.exists(roc_path):
    axes[1].imshow(plt.imread(roc_path))
    axes[1].set_title("Metal Nut ROC Curve")
axes[1].axis('off')

if os.path.exists(cm_path):
    axes[2].imshow(plt.imread(cm_path))
    axes[2].set_title("Metal Nut Confusion Matrix")
axes[2].axis('off')

plt.tight_layout()
plt.show()

---
## 7. BẢNG TỔNG HỢP SO SÁNH ĐA DANH MỤC (MULTI-CLASS BENCHMARK)

Tổng kết năng lực phát hiện và khoanh vùng bất thường trên cả 3 dạng bề mặt công nghiệp: Thủy tinh (`bottle`), Dược phẩm (`capsule`), Cơ khí (`metal_nut`).

In [ ]:
import json
import os
import pandas as pd
from IPython.display import display

experiments = {
    'Bottle (Thủy tinh)': '/content/drive/MyDrive/CAE_Project/outputs/E2_bottle_l1_bottleneck_seed42/metrics/bottle_metrics.json',
    'Capsule (Dược phẩm)': '/content/drive/MyDrive/CAE_Project/outputs/E2_capsule_l1_bottleneck_seed42/metrics/capsule_metrics.json',
    'Metal Nut (Cơ khí)': '/content/drive/MyDrive/CAE_Project/outputs/E2_metal_nut_l1_bottleneck_seed42/metrics/metal_nut_metrics.json'
}

rows = []
for cat_name, m_path in experiments.items():
    if os.path.exists(m_path):
        with open(m_path) as f:
            data = json.load(f)
        im = data['image_level']
        px = data['pixel_level']
        rows.append({
            'Danh mục sản phẩm': cat_name,
            'Image AUROC': im['auroc'],
            'Image F1': im['f1'],
            'Precision': im['precision'],
            'Recall': im['recall'],
            'Pixel AUROC': px['pixel_auroc'],
            'Pixel IoU': px['pixel_iou'],
            'Pixel Dice': px['pixel_dice']
        })

if rows:
    df_bench = pd.DataFrame(rows)
    # Tính dòng trung bình (Mean Average)
    numeric_cols = ['Image AUROC', 'Image F1', 'Precision', 'Recall', 'Pixel AUROC', 'Pixel IoU', 'Pixel Dice']
    mean_row = {'Danh mục sản phẩm': 'TRUNG BÌNH (MEAN)'}
    for col in numeric_cols:
        mean_row[col] = df_bench[col].mean()
    df_bench = pd.concat([df_bench, pd.DataFrame([mean_row])], ignore_index=True)
    
    # Định dạng hiển thị 4 chữ số thập phân
    for col in numeric_cols:
        df_bench[col] = df_bench[col].apply(lambda x: f"{x:.4f}")
        
    print("=========================================================================================")
    print("             BẢNG ĐÁNH GIÁ NĂNG LỰC CAE TRÊN NHIỀU BỀ MẶT CÔNG NGHIỆP (MVTEC AD)         ")
    print("=========================================================================================")
    display(df_bench)
else:
    print("Chưa có kết quả để hiển thị. Hãy chạy các cell huấn luyện bên trên!")